# 06. SCARF Baseline & Robustness

This notebook implements Self-Supervised Contrastive Learning (SCARF) for tabular network traffic data, conducts multi-seed reproducibility benchmarks, tests robustness under dTTL perturbations, and evaluates statistical significance via paired-bootstrap tests.


In [ ]:
# ================================================================
# SCARF — PILOT IMPLEMENTATION
# ================================================================

import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset


print("=" * 80)
print("SCARF PILOT — CONTRASTIVE REPRESENTATION LEARNING")
print("=" * 80)


# ================================================================
# 1. REPRODUCIBILITY
# ================================================================

SEED = 42

def set_seed_scarf(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


set_seed_scarf(SEED)


# ================================================================
# 2. DEVICE
# ================================================================

device_scarf = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device_scarf)


# ================================================================
# 3. PREPARE DATA
# ================================================================

TARGET = "label"
CATEGORY = "attack_cat"

X_train_raw_s = train_df.drop(
    columns=[TARGET, CATEGORY],
    errors="ignore"
).copy()

X_test_raw_s = test_df.drop(
    columns=[TARGET, CATEGORY],
    errors="ignore"
).copy()

y_train_s = (
    train_df[TARGET]
    .astype(int)
    .to_numpy()
)

y_test_s = (
    test_df[TARGET]
    .astype(int)
    .to_numpy()
)


# ================================================================
# 4. SAME FITTED PREPROCESSOR
# ================================================================

X_train_s = preprocessor.transform(
    X_train_raw_s
)

X_test_s = preprocessor.transform(
    X_test_raw_s
)

# Convert sparse matrix if necessary
if hasattr(X_train_s, "toarray"):
    X_train_s = X_train_s.toarray()

if hasattr(X_test_s, "toarray"):
    X_test_s = X_test_s.toarray()


X_train_s = np.asarray(
    X_train_s,
    dtype=np.float32
)

X_test_s = np.asarray(
    X_test_s,
    dtype=np.float32
)


print(
    "Training shape:",
    X_train_s.shape
)

print(
    "Test shape:",
    X_test_s.shape
)


# ================================================================
# 5. SCARF PARAMETERS
# ================================================================

INPUT_DIM = X_train_s.shape[1]

EMBED_DIM = 64

HIDDEN_DIM = 128

CORRUPTION_RATE = 0.30

TEMPERATURE = 0.20

CONTRASTIVE_EPOCHS = 10

FINETUNE_EPOCHS = 10

BATCH_SIZE = 2048

LEARNING_RATE = 1e-3


print("\nSCARF configuration:")
print("Input dimension:", INPUT_DIM)
print("Embedding dimension:", EMBED_DIM)
print("Corruption rate:", CORRUPTION_RATE)
print("Temperature:", TEMPERATURE)


# ================================================================
# 6. SCARF ENCODER
# ================================================================

class SCARFEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        embed_dim=64
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                embed_dim
            )
        )

    def forward(self, x):

        z = self.network(x)

        # Normalize representation
        z = F.normalize(
            z,
            dim=1
        )

        return z


# ================================================================
# 7. SCARF FEATURE CORRUPTION
# ================================================================

def scarf_corrupt_batch(
    x,
    corruption_rate
):

    """
    SCARF-style random feature corruption.

    Each sample receives a corrupted view where a subset
    of feature values is replaced using values drawn from
    other samples in the batch.

    This corruption is used ONLY for representation learning.
    It is NOT the TTL robustness transformation.
    """

    batch_size, num_features = x.shape

    corrupted = x.clone()

    mask = (
        torch.rand_like(x)
        < corruption_rate
    )

    # Random donor rows
    donor_indices = torch.randperm(
        batch_size,
        device=x.device
    )

    donor_values = x[
        donor_indices
    ]

    corrupted[mask] = donor_values[mask]

    return corrupted


# ================================================================
# 8. CONTRASTIVE LOSS
# ================================================================

def scarf_contrastive_loss(
    z1,
    z2,
    temperature=0.20
):

    """
    NT-Xent / InfoNCE-style loss.

    Positive pair:
        original sample ↔ corrupted view

    Negatives:
        other samples in the batch
    """

    similarity = torch.matmul(
        z1,
        z2.T
    ) / temperature

    labels = torch.arange(
        z1.size(0),
        device=z1.device
    )

    loss_12 = F.cross_entropy(
        similarity,
        labels
    )

    loss_21 = F.cross_entropy(
        similarity.T,
        labels
    )

    return (
        loss_12 + loss_21
    ) / 2


# ================================================================
# 9. CREATE ENCODER
# ================================================================

encoder_scarf = SCARFEncoder(
    input_dim=INPUT_DIM,
    hidden_dim=HIDDEN_DIM,
    embed_dim=EMBED_DIM
).to(device_scarf)


# ================================================================
# 10. CONTRASTIVE PRETRAINING
# ================================================================

train_tensor_s = torch.tensor(
    X_train_s,
    dtype=torch.float32
)

contrastive_loader = DataLoader(
    TensorDataset(train_tensor_s),
    batch_size=BATCH_SIZE,
    shuffle=True,
    drop_last=True
)


optimizer_scarf = torch.optim.Adam(
    encoder_scarf.parameters(),
    lr=LEARNING_RATE
)


print("\n")
print("=" * 80)
print("SCARF CONTRASTIVE PRETRAINING")
print("=" * 80)


encoder_scarf.train()

for epoch in range(
    CONTRASTIVE_EPOCHS
):

    total_loss = 0.0

    for (xb,) in contrastive_loader:

        xb = xb.to(
            device_scarf
        )

        # Original representation
        z1 = encoder_scarf(
            xb
        )

        # Corrupted representation
        xb_corrupted = scarf_corrupt_batch(
            xb,
            CORRUPTION_RATE
        )

        z2 = encoder_scarf(
            xb_corrupted
        )

        # Contrastive loss
        loss = scarf_contrastive_loss(
            z1,
            z2,
            TEMPERATURE
        )

        optimizer_scarf.zero_grad()

        loss.backward()

        optimizer_scarf.step()

        total_loss += loss.item()

    mean_loss = (
        total_loss /
        len(contrastive_loader)
    )

    print(
        f"Epoch {epoch + 1:02d}/"
        f"{CONTRASTIVE_EPOCHS} "
        f"| Contrastive Loss: "
        f"{mean_loss:.6f}"
    )


# ================================================================
# 11. CLASSIFICATION HEAD
# ================================================================

class SCARFClassifier(nn.Module):

    def __init__(
        self,
        encoder,
        embed_dim
    ):

        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Linear(
            embed_dim,
            1
        )

    def forward(self, x):

        z = self.encoder(x)

        return self.classifier(
            z
        ).squeeze(1)


scarf_model = SCARFClassifier(
    encoder_scarf,
    EMBED_DIM
).to(device_scarf)


# ================================================================
# 12. SUPERVISED FINE-TUNING
# ================================================================

y_tensor_s = torch.tensor(
    y_train_s,
    dtype=torch.float32
)

supervised_dataset = TensorDataset(
    train_tensor_s,
    y_tensor_s
)

supervised_loader = DataLoader(
    supervised_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)


# Class imbalance handling
positives = y_train_s.sum()
negatives = len(y_train_s) - positives

pos_weight = torch.tensor(
    [negatives / positives],
    dtype=torch.float32,
    device=device_scarf
)


criterion_scarf = nn.BCEWithLogitsLoss(
    pos_weight=pos_weight
)


optimizer_finetune = torch.optim.Adam(
    scarf_model.parameters(),
    lr=LEARNING_RATE
)


print("\n")
print("=" * 80)
print("SCARF SUPERVISED FINE-TUNING")
print("=" * 80)


for epoch in range(
    FINETUNE_EPOCHS
):

    scarf_model.train()

    total_loss = 0.0

    for xb, yb in supervised_loader:

        xb = xb.to(
            device_scarf
        )

        yb = yb.to(
            device_scarf
        )

        logits = scarf_model(
            xb
        )

        loss = criterion_scarf(
            logits,
            yb
        )

        optimizer_finetune.zero_grad()

        loss.backward()

        optimizer_finetune.step()

        total_loss += loss.item()

    mean_loss = (
        total_loss /
        len(supervised_loader)
    )

    print(
        f"Epoch {epoch + 1:02d}/"
        f"{FINETUNE_EPOCHS} "
        f"| Classification Loss: "
        f"{mean_loss:.6f}"
    )


# ================================================================
# 13. EVALUATION
# ================================================================

from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)


def evaluate_scarf(
    model,
    X,
    y
):

    model.eval()

    with torch.no_grad():

        X_tensor = torch.tensor(
            X,
            dtype=torch.float32,
            device=device_scarf
        )

        logits = model(
            X_tensor
        )

        probability = (
            torch.sigmoid(logits)
            .cpu()
            .numpy()
        )

    prediction = (
        probability >= 0.5
    ).astype(int)

    tpr = recall_score(
        y,
        prediction,
        zero_division=0
    )

    auroc = roc_auc_score(
        y,
        probability
    )

    auprc = average_precision_score(
        y,
        probability
    )

    return (
        tpr,
        auroc,
        auprc
    )


clean_tpr, clean_auc, clean_ap = (
    evaluate_scarf(
        scarf_model,
        X_test_s,
        y_test_s
    )
)


print("\n")
print("=" * 80)
print("SCARF CLEAN TEST PERFORMANCE")
print("=" * 80)

print(
    f"TPR   : {clean_tpr:.6f}"
)

print(
    f"AUROC : {clean_auc:.6f}"
)

print(
    f"AUPRC : {clean_ap:.6f}"
)


# ================================================================
# 14. dTTL ROBUSTNESS TEST
# ================================================================

print("\n")
print("=" * 80)
print("SCARF dTTL ROBUSTNESS TEST")
print("=" * 80)


dttl_results = []


for percent in [
    -15, -10, -5, -2,
     0,
     2, 5, 10, 15
]:

    if percent == 0:

        X_transformed = X_test_s

    else:

        transformed_raw = X_test_raw_s.copy()

        values = (
            transformed_raw["dttl"]
            .astype(float)
            .to_numpy()
        )

        transformed = (
            values *
            (1 + percent / 100.0)
        )

        transformed = np.round(
            transformed
        )

        transformed = np.clip(
            transformed,
            1,
            255
        )

        transformed_raw["dttl"] = (
            transformed
        )

        X_transformed = (
            preprocessor.transform(
                transformed_raw
            )
        )

        if hasattr(
            X_transformed,
            "toarray"
        ):

            X_transformed = (
                X_transformed.toarray()
            )

        X_transformed = np.asarray(
            X_transformed,
            dtype=np.float32
        )


    tpr, auc, ap = evaluate_scarf(
        scarf_model,
        X_transformed,
        y_test_s
    )

    drop = (
        tpr -
        clean_tpr
    )

    dttl_results.append({

        "dTTL Perturbation (%)":
            percent,

        "TPR":
            tpr,

        "TPR Drop":
            drop,

        "AUROC":
            auc,

        "AUPRC":
            ap
    })


scarf_dttl_results = pd.DataFrame(
    dttl_results
)


print(
    scarf_dttl_results.round(6)
    .to_string(index=False)
)


# ================================================================
# 15. SAVE PILOT RESULTS
# ================================================================

scarf_dttl_results.to_csv(
    "scarf_pilot_dttl_results.csv",
    index=False
)


print("\n")
print("=" * 80)
print("SCARF PILOT COMPLETE")
print("=" * 80)

print(
    "Saved: scarf_pilot_dttl_results.csv"
)

SCARF PILOT — CONTRASTIVE REPRESENTATION LEARNING
Device: cuda
Training shape: (175341, 195)
Test shape: (82332, 195)

SCARF configuration:
Input dimension: 195
Embedding dimension: 64
Corruption rate: 0.3
Temperature: 0.2


SCARF CONTRASTIVE PRETRAINING
Epoch 01/10 | Contrastive Loss: 5.405734
Epoch 02/10 | Contrastive Loss: 5.058869
Epoch 03/10 | Contrastive Loss: 4.892148
Epoch 04/10 | Contrastive Loss: 4.784254
Epoch 05/10 | Contrastive Loss: 4.710740
Epoch 06/10 | Contrastive Loss: 4.667235
Epoch 07/10 | Contrastive Loss: 4.620961
Epoch 08/10 | Contrastive Loss: 4.594201
Epoch 09/10 | Contrastive Loss: 4.564400
Epoch 10/10 | Contrastive Loss: 4.551711


SCARF SUPERVISED FINE-TUNING
Epoch 01/10 | Classification Loss: 0.280537
Epoch 02/10 | Classification Loss: 0.173942
Epoch 03/10 | Classification Loss: 0.133044
Epoch 04/10 | Classification Loss: 0.110068
Epoch 05/10 | Classification Loss: 0.095112
Epoch 06/10 | Classification Loss: 0.083669
Epoch 07/10 | Classification Loss: 0.075

In [ ]:
# ================================================================
# SCARF — 5 SEED REPRODUCIBILITY EXPERIMENT
# ================================================================

import numpy as np
import pandas as pd
import random
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import (
    recall_score,
    roc_auc_score,
    average_precision_score
)

# ------------------------------------------------
# SETTINGS
# ------------------------------------------------

SEEDS = [42, 123, 2026]

DTTL_LEVELS = [
    -15, -10, -5, -2,
      0,
       2,  5, 10, 15
]

BATCH_SIZE = 2048

EMBED_DIM = 64
HIDDEN_DIM = 128

CORRUPTION_RATE = 0.30
TEMPERATURE = 0.20

CONTRASTIVE_EPOCHS = 10
FINETUNE_EPOCHS = 10

LEARNING_RATE = 1e-3

device_scarf_multi = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

print("=" * 80)
print("SCARF — 5 SEED REPRODUCIBILITY EXPERIMENT")
print("=" * 80)

print("Device:", device_scarf_multi)
print("Seeds:", SEEDS)
print("dTTL levels:", DTTL_LEVELS)


# ================================================================
# 1. REPRODUCIBILITY
# ================================================================

def set_seed_multi(seed):

    random.seed(seed)
    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


# ================================================================
# 2. SCARF ENCODER
# ================================================================

class SCARFEncoderMulti(nn.Module):

    def __init__(
        self,
        input_dim,
        hidden_dim=128,
        embed_dim=64
    ):

        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim
            ),

            nn.ReLU(),

            nn.Linear(
                hidden_dim,
                embed_dim
            )
        )

    def forward(self, x):

        z = self.network(x)

        return F.normalize(
            z,
            dim=1
        )


# ================================================================
# 3. SCARF CLASSIFIER
# ================================================================

class SCARFClassifierMulti(nn.Module):

    def __init__(
        self,
        encoder,
        embed_dim
    ):

        super().__init__()

        self.encoder = encoder

        self.classifier = nn.Linear(
            embed_dim,
            1
        )

    def forward(self, x):

        z = self.encoder(x)

        return self.classifier(
            z
        ).squeeze(1)


# ================================================================
# 4. SCARF CORRUPTION
# ================================================================

def scarf_corrupt_batch_multi(
    x,
    corruption_rate
):

    batch_size, num_features = x.shape

    corrupted = x.clone()

    mask = (
        torch.rand_like(x)
        < corruption_rate
    )

    donor_indices = torch.randperm(
        batch_size,
        device=x.device
    )

    donor_values = x[
        donor_indices
    ]

    corrupted[mask] = donor_values[mask]

    return corrupted


# ================================================================
# 5. CONTRASTIVE LOSS
# ================================================================

def scarf_loss_multi(
    z1,
    z2,
    temperature
):

    similarity = (
        torch.matmul(
            z1,
            z2.T
        )
        / temperature
    )

    labels = torch.arange(
        z1.size(0),
        device=z1.device
    )

    loss_12 = F.cross_entropy(
        similarity,
        labels
    )

    loss_21 = F.cross_entropy(
        similarity.T,
        labels
    )

    return (
        loss_12 + loss_21
    ) / 2


# ================================================================
# 6. EVALUATION
# ================================================================

def evaluate_scarf_multi(
    model,
    X,
    y
):

    model.eval()

    with torch.no_grad():

        X_tensor = torch.tensor(
            X,
            dtype=torch.float32,
            device=device_scarf_multi
        )

        logits = model(
            X_tensor
        )

        probability = (
            torch.sigmoid(logits)
            .cpu()
            .numpy()
        )

    prediction = (
        probability >= 0.5
    ).astype(int)

    tpr = recall_score(
        y,
        prediction,
        zero_division=0
    )

    auroc = roc_auc_score(
        y,
        probability
    )

    auprc = average_precision_score(
        y,
        probability
    )

    return tpr, auroc, auprc


# ================================================================
# 7. BUILD dTTL TEST SET
# ================================================================

def make_dttl_test_set(
    percent
):

    if percent == 0:

        X_transformed = X_test_s

    else:

        transformed_raw = (
            X_test_raw_s.copy()
        )

        values = (
            transformed_raw["dttl"]
            .astype(float)
            .to_numpy()
        )

        transformed = (
            values *
            (1 + percent / 100.0)
        )

        # dttl is integer-valued in UNSW-NB15
        transformed = np.round(
            transformed
        )

        # Keep technically valid byte-range values
        transformed = np.clip(
            transformed,
            1,
            255
        )

        transformed_raw["dttl"] = (
            transformed
        )

        X_transformed = (
            preprocessor.transform(
                transformed_raw
            )
        )

        if hasattr(
            X_transformed,
            "toarray"
        ):

            X_transformed = (
                X_transformed.toarray()
            )

        X_transformed = np.asarray(
            X_transformed,
            dtype=np.float32
        )

    return X_transformed


# ================================================================
# 8. PRE-COMPUTE TEST SETS
# ================================================================

print("\nPreparing dTTL test sets...")

dttl_test_sets = {}

for level in DTTL_LEVELS:

    dttl_test_sets[level] = (
        make_dttl_test_set(level)
    )

    print(
        f"dTTL {level:+d}% -> "
        f"{dttl_test_sets[level].shape}"
    )


# ================================================================
# 9. TRAIN ONE SCARF MODEL
# ================================================================

def train_one_scarf(seed):

    print("\n")
    print("=" * 80)
    print(f"SCARF SEED {seed}")
    print("=" * 80)

    set_seed_multi(seed)

    # ------------------------------------------------
    # Fresh encoder
    # ------------------------------------------------

    encoder = SCARFEncoderMulti(
        input_dim=X_train_s.shape[1],
        hidden_dim=HIDDEN_DIM,
        embed_dim=EMBED_DIM
    ).to(device_scarf_multi)

    # ------------------------------------------------
    # Training tensor
    # ------------------------------------------------

    X_tensor = torch.tensor(
        X_train_s,
        dtype=torch.float32
    )

    y_tensor = torch.tensor(
        y_train_s,
        dtype=torch.float32
    )

    contrastive_loader = DataLoader(
        TensorDataset(X_tensor),
        batch_size=BATCH_SIZE,
        shuffle=True,
        drop_last=True
    )

    # ------------------------------------------------
    # Contrastive pretraining
    # ------------------------------------------------

    optimizer = torch.optim.Adam(
        encoder.parameters(),
        lr=LEARNING_RATE
    )

    encoder.train()

    for epoch in range(
        CONTRASTIVE_EPOCHS
    ):

        total_loss = 0.0

        for (xb,) in contrastive_loader:

            xb = xb.to(
                device_scarf_multi
            )

            z1 = encoder(xb)

            xb_corrupted = (
                scarf_corrupt_batch_multi(
                    xb,
                    CORRUPTION_RATE
                )
            )

            z2 = encoder(
                xb_corrupted
            )

            loss = scarf_loss_multi(
                z1,
                z2,
                TEMPERATURE
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        if (
            epoch == 0
            or epoch == CONTRASTIVE_EPOCHS - 1
        ):

            print(
                f"Contrastive epoch "
                f"{epoch + 1}/"
                f"{CONTRASTIVE_EPOCHS} "
                f"| Loss = "
                f"{total_loss / len(contrastive_loader):.6f}"
            )

    # ------------------------------------------------
    # Classification model
    # ------------------------------------------------

    model = SCARFClassifierMulti(
        encoder,
        EMBED_DIM
    ).to(device_scarf_multi)

    supervised_loader = DataLoader(
        TensorDataset(
            X_tensor,
            y_tensor
        ),
        batch_size=BATCH_SIZE,
        shuffle=True
    )

    positives = y_train_s.sum()

    negatives = (
        len(y_train_s)
        - positives
    )

    pos_weight = torch.tensor(
        [negatives / positives],
        dtype=torch.float32,
        device=device_scarf_multi
    )

    criterion = (
        nn.BCEWithLogitsLoss(
            pos_weight=pos_weight
        )
    )

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LEARNING_RATE
    )

    # ------------------------------------------------
    # Fine-tuning
    # ------------------------------------------------

    for epoch in range(
        FINETUNE_EPOCHS
    ):

        model.train()

        total_loss = 0.0

        for xb, yb in supervised_loader:

            xb = xb.to(
                device_scarf_multi
            )

            yb = yb.to(
                device_scarf_multi
            )

            logits = model(xb)

            loss = criterion(
                logits,
                yb
            )

            optimizer.zero_grad()

            loss.backward()

            optimizer.step()

            total_loss += loss.item()

        if (
            epoch == 0
            or epoch == FINETUNE_EPOCHS - 1
        ):

            print(
                f"Fine-tune epoch "
                f"{epoch + 1}/"
                f"{FINETUNE_EPOCHS} "
                f"| Loss = "
                f"{total_loss / len(supervised_loader):.6f}"
            )

    # ------------------------------------------------
    # Evaluate
    # ------------------------------------------------

    clean_tpr, clean_auc, clean_ap = (
        evaluate_scarf_multi(
            model,
            X_test_s,
            y_test_s
        )
    )

    print(
        f"\nClean -> "
        f"TPR={clean_tpr:.6f}, "
        f"AUROC={clean_auc:.6f}, "
        f"AUPRC={clean_ap:.6f}"
    )

    results = []

    for level in DTTL_LEVELS:

        tpr, auc, ap = (
            evaluate_scarf_multi(
                model,
                dttl_test_sets[level],
                y_test_s
            )
        )

        results.append({

            "Seed": seed,

            "dTTL (%)": level,

            "TPR": tpr,

            "TPR Drop": (
                tpr - clean_tpr
            ),

            "AUROC": auc,

            "AUPRC": ap

        })

    return pd.DataFrame(
        results
    )


# ================================================================
# 10. RUN ALL FIVE SEEDS
# ================================================================

all_scarf_results = []

for seed in SEEDS:

    seed_results = train_one_scarf(
        seed
    )

    all_scarf_results.append(
        seed_results
    )


scarf_5seed_results = pd.concat(
    all_scarf_results,
    ignore_index=True
)


# ================================================================
# 11. SAVE RAW RESULTS
# ================================================================

scarf_5seed_results.to_csv(
    "scarf_dttl_5seed_results.csv",
    index=False
)


# ================================================================
# 12. MEAN ± SD
# ================================================================

summary = (
    scarf_5seed_results
    .groupby("dTTL (%)")
    .agg({

        "TPR": ["mean", "std"],

        "TPR Drop": ["mean", "std"],

        "AUROC": ["mean", "std"],

        "AUPRC": ["mean", "std"]

    })
)

summary.columns = [
    "TPR Mean",
    "TPR SD",
    "TPR Drop Mean",
    "TPR Drop SD",
    "AUROC Mean",
    "AUROC SD",
    "AUPRC Mean",
    "AUPRC SD"
]

summary = summary.reset_index()


# ================================================================
# 13. DISPLAY
# ================================================================

print("\n")
print("=" * 100)
print("SCARF — 5 SEED dTTL ROBUSTNESS SUMMARY")
print("=" * 100)

print(
    summary.round(6)
    .to_string(index=False)
)


# ================================================================
# 14. SAVE SUMMARY
# ================================================================

summary.to_csv(
    "scarf_dttl_5seed_summary.csv",
    index=False
)

print("\n")
print("=" * 80)
print("5-SEED SCARF EXPERIMENT COMPLETE")
print("=" * 80)

print(
    "Saved:"
)

print(
    "  scarf_dttl_5seed_results.csv"
)

print(
    "  scarf_dttl_5seed_summary.csv"
)

SCARF — 5 SEED REPRODUCIBILITY EXPERIMENT
Device: cuda
Seeds: [42, 123, 2026]
dTTL levels: [-15, -10, -5, -2, 0, 2, 5, 10, 15]

Preparing dTTL test sets...
dTTL -15% -> (82332, 195)
dTTL -10% -> (82332, 195)
dTTL -5% -> (82332, 195)
dTTL -2% -> (82332, 195)
dTTL +0% -> (82332, 195)
dTTL +2% -> (82332, 195)
dTTL +5% -> (82332, 195)
dTTL +10% -> (82332, 195)
dTTL +15% -> (82332, 195)


SCARF SEED 42
Contrastive epoch 1/10 | Loss = 5.405734
Contrastive epoch 10/10 | Loss = 4.551711
Fine-tune epoch 1/10 | Loss = 0.280537
Fine-tune epoch 10/10 | Loss = 0.056147

Clean -> TPR=0.461969, AUROC=0.784007, AUPRC=0.820458


SCARF SEED 123
Contrastive epoch 1/10 | Loss = 5.382530
Contrastive epoch 10/10 | Loss = 4.548234
Fine-tune epoch 1/10 | Loss = 0.282786
Fine-tune epoch 10/10 | Loss = 0.053073

Clean -> TPR=0.450234, AUROC=0.766271, AUPRC=0.803756


SCARF SEED 2026
Contrastive epoch 1/10 | Loss = 5.396891
Contrastive epoch 10/10 | Loss = 4.543354
Fine-tune epoch 1/10 | Loss = 0.272726
Fine-tun

In [ ]:
# ================================================================
# M6 — STANDARD MLP vs TTL-AUGMENTED MLP vs SCARF
#      SAME dTTL ROBUSTNESS TEST
# ================================================================

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

print("=" * 80)
print("M6 — THREE-WAY ROBUSTNESS COMPARISON")
print("=" * 80)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)

# ------------------------------------------------
# 1. PERTURBATION LEVELS
# ------------------------------------------------

PERTURBATIONS = [-15, -10, -5, -2, 0, 2, 5, 10, 15]

# ------------------------------------------------
# 2. CHECK DATA
# ------------------------------------------------

if "test_df" not in globals():
    raise RuntimeError("test_df not found.")

if "preprocessor" not in globals():
    raise RuntimeError("preprocessor not found.")

if "label" not in test_df.columns:
    raise RuntimeError("'label' column not found.")

y_test = test_df["label"].astype(int).values

print("Test samples:", len(y_test))


# ------------------------------------------------
# 3. IDENTIFY EXISTING MODELS
# ------------------------------------------------

print("\nAVAILABLE MODEL VARIABLES")
print("-" * 50)

possible_models = [
    "standard_model",
    "augmented_model",
    "model_aug",
    "scarf_model",
    "model_scarf",
    "scarf",
    "model"
]

for name in possible_models:

    if name in globals():

        obj = globals()[name]

        if isinstance(obj, torch.nn.Module):

            print(name, "->", type(obj).__name__)


# ------------------------------------------------
# 4. FIND STANDARD MODEL
# ------------------------------------------------

standard_candidates = [
    "standard_model",
    "model_clean",
    "clean_model"
]

standard_model_found = None

for name in standard_candidates:

    if name in globals():

        if isinstance(globals()[name], torch.nn.Module):

            standard_model_found = globals()[name]
            print("\nStandard MLP:", name)
            break

if standard_model_found is None:

    raise RuntimeError(
        "\nStandard MLP not found.\n"
        "Expected variable: standard_model\n"
        "Please make sure the clean MLP model is still in memory."
    )


# ------------------------------------------------
# 5. FIND AUGMENTED MODEL
# ------------------------------------------------

aug_candidates = [
    "augmented_model",
    "model_aug",
    "aug_model"
]

augmented_model_found = None

for name in aug_candidates:

    if name in globals():

        if isinstance(globals()[name], torch.nn.Module):

            augmented_model_found = globals()[name]
            print("Augmented MLP:", name)
            break

if augmented_model_found is None:

    raise RuntimeError(
        "\nAugmented MLP not found.\n"
        "Expected variable: augmented_model or model_aug."
    )


# ------------------------------------------------
# 6. FIND SCARF MODEL
# ------------------------------------------------

scarf_candidates = [
    "scarf_model",
    "model_scarf",
    "scarf",
    "model"
]

scarf_model_found = None
scarf_name = None

for name in scarf_candidates:

    if name in globals():

        if isinstance(globals()[name], torch.nn.Module):

            scarf_model_found = globals()[name]
            scarf_name = name
            break

if scarf_model_found is None:

    raise RuntimeError(
        "\nSCARF model not found.\n"
        "Expected variable: scarf_model, model_scarf, scarf or model."
    )

print("SCARF model:", scarf_name)


# ------------------------------------------------
# 7. PUT MODELS ON SAME DEVICE
# ------------------------------------------------

standard_model_found = standard_model_found.to(DEVICE)
augmented_model_found = augmented_model_found.to(DEVICE)
scarf_model_found = scarf_model_found.to(DEVICE)


# ------------------------------------------------
# 8. CREATE CLEAN PROCESSED TEST DATA
# ------------------------------------------------

test_raw = test_df.drop(
    columns=["label"]
).copy()

X_test_clean = preprocessor.transform(test_raw)

if hasattr(X_test_clean, "toarray"):
    X_test_clean = X_test_clean.toarray()

X_test_clean = np.asarray(
    X_test_clean,
    dtype=np.float32
)

print("\nProcessed test shape:", X_test_clean.shape)


# ------------------------------------------------
# 9. PREDICTION FUNCTION
# ------------------------------------------------

def predict_probability(model, X):

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    ).to(DEVICE)

    predictions = []

    with torch.no_grad():

        for start in range(
            0,
            len(X_tensor),
            4096
        ):

            batch = X_tensor[
                start:start + 4096
            ]

            output = model(batch)

            # Handle models returning shape [N,1]
            if output.ndim > 1:
                output = output.squeeze(-1)

            probability = torch.sigmoid(output)

            predictions.append(
                probability.detach().cpu().numpy()
            )

    return np.concatenate(predictions)


# ------------------------------------------------
# 10. METRIC FUNCTION
# ------------------------------------------------

def get_metrics(y_true, probability):

    prediction = (
        probability >= 0.5
    ).astype(int)

    tp = np.sum(
        (y_true == 1) &
        (prediction == 1)
    )

    fn = np.sum(
        (y_true == 1) &
        (prediction == 0)
    )

    fp = np.sum(
        (y_true == 0) &
        (prediction == 1)
    )

    tn = np.sum(
        (y_true == 0) &
        (prediction == 0)
    )

    tpr = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else 0
    )

    fpr = (
        fp / (fp + tn)
        if (fp + tn) > 0
        else 0
    )

    return {
        "TPR": tpr,
        "FPR": fpr,
        "AUROC": roc_auc_score(
            y_true,
            probability
        ),
        "AUPRC": average_precision_score(
            y_true,
            probability
        )
    }


# ------------------------------------------------
# 11. TEST PERTURBATION FUNCTION
# ------------------------------------------------

def make_dttl_test(percent):

    df = test_df.drop(
        columns=["label"]
    ).copy()

    if percent != 0:

        factor = 1.0 + (
            percent / 100.0
        )

        dttl_values = pd.to_numeric(
            df["dttl"],
            errors="coerce"
        ).astype(float)

        transformed = np.round(
            dttl_values * factor
        )

        # Technically valid TTL range
        transformed = np.clip(
            transformed,
            1,
            255
        )

        df["dttl"] = transformed

    X = preprocessor.transform(df)

    if hasattr(X, "toarray"):
        X = X.toarray()

    return np.asarray(
        X,
        dtype=np.float32
    )


# ================================================================
# 12. RUN THREE-WAY ROBUSTNESS TEST
# ================================================================

results = []

print("\n" + "=" * 80)
print("RUNNING THREE-WAY dTTL ROBUSTNESS TEST")
print("=" * 80)

for pct in PERTURBATIONS:

    print(
        f"\nTesting dTTL {pct:+d}% ..."
    )

    X_test = make_dttl_test(pct)

    # ----------------------------
    # STANDARD MLP
    # ----------------------------

    std_prob = predict_probability(
        standard_model_found,
        X_test
    )

    std = get_metrics(
        y_test,
        std_prob
    )

    # ----------------------------
    # TTL AUGMENTED MLP
    # ----------------------------

    aug_prob = predict_probability(
        augmented_model_found,
        X_test
    )

    aug = get_metrics(
        y_test,
        aug_prob
    )

    # ----------------------------
    # SCARF
    # ----------------------------

    scarf_prob = predict_probability(
        scarf_model_found,
        X_test
    )

    scarf = get_metrics(
        y_test,
        scarf_prob
    )

    results.append({

        "dTTL_%": pct,

        "Standard_TPR": std["TPR"],
        "Standard_FPR": std["FPR"],
        "Standard_AUROC": std["AUROC"],
        "Standard_AUPRC": std["AUPRC"],

        "Augmented_TPR": aug["TPR"],
        "Augmented_FPR": aug["FPR"],
        "Augmented_AUROC": aug["AUROC"],
        "Augmented_AUPRC": aug["AUPRC"],

        "SCARF_TPR": scarf["TPR"],
        "SCARF_FPR": scarf["FPR"],
        "SCARF_AUROC": scarf["AUROC"],
        "SCARF_AUPRC": scarf["AUPRC"]
    })


# ------------------------------------------------
# 13. RESULTS TABLE
# ------------------------------------------------

m6_results = pd.DataFrame(results)

pd.set_option(
    "display.max_columns",
    None
)

pd.set_option(
    "display.width",
    300
)

print("\n" + "=" * 80)
print("M6 — THREE-WAY dTTL ROBUSTNESS RESULTS")
print("=" * 80)

print(
    m6_results.round(6).to_string(
        index=False
    )
)


# ================================================================
# 14. CALCULATE TPR DROP FROM CLEAN
# ================================================================

clean_row = m6_results[
    m6_results["dTTL_%"] == 0
].iloc[0]

m6_results["Standard_TPR_Drop"] = (
    m6_results["Standard_TPR"]
    - clean_row["Standard_TPR"]
)

m6_results["Augmented_TPR_Drop"] = (
    m6_results["Augmented_TPR"]
    - clean_row["Augmented_TPR"]
)

m6_results["SCARF_TPR_Drop"] = (
    m6_results["SCARF_TPR"]
    - clean_row["SCARF_TPR"]
)


# ------------------------------------------------
# 15. FINAL ROBUSTNESS TABLE
# ------------------------------------------------

print("\n" + "=" * 80)
print("TPR ROBUSTNESS COMPARISON")
print("=" * 80)

print(
    m6_results[
        [
            "dTTL_%",
            "Standard_TPR",
            "Standard_TPR_Drop",
            "Augmented_TPR",
            "Augmented_TPR_Drop",
            "SCARF_TPR",
            "SCARF_TPR_Drop"
        ]
    ].round(6).to_string(index=False)
)


# ------------------------------------------------
# 16. SAVE
# ------------------------------------------------

m6_results.to_csv(
    "M6_three_way_dttl_robustness.csv",
    index=False
)

print("\nSaved:")
print("M6_three_way_dttl_robustness.csv")

print("\n" + "=" * 80)
print("M6 COMPLETE")
print("=" * 80)

M6 — THREE-WAY ROBUSTNESS COMPARISON
Device: cuda
Test samples: 82332

AVAILABLE MODEL VARIABLES
--------------------------------------------------
standard_model -> MLP
augmented_model -> MLP
scarf_model -> SCARFClassifier
model -> MLP

Standard MLP: standard_model
Augmented MLP: augmented_model
SCARF model: scarf_model

Processed test shape: (82332, 195)

RUNNING THREE-WAY dTTL ROBUSTNESS TEST

Testing dTTL -15% ...

Testing dTTL -10% ...

Testing dTTL -5% ...

Testing dTTL -2% ...

Testing dTTL +0% ...

Testing dTTL +2% ...

Testing dTTL +5% ...

Testing dTTL +10% ...

Testing dTTL +15% ...

M6 — THREE-WAY dTTL ROBUSTNESS RESULTS
 dTTL_%  Standard_TPR  Standard_FPR  Standard_AUROC  Standard_AUPRC  Augmented_TPR  Augmented_FPR  Augmented_AUROC  Augmented_AUPRC  SCARF_TPR  SCARF_FPR  SCARF_AUROC  SCARF_AUPRC
    -15      0.392328      0.232784        0.633283        0.732423       0.429233       0.293730         0.567638         0.721184   0.423785   0.153649     0.776913     0.812448

In [ ]:
# ================================================================
# M7 — STATISTICAL SIGNIFICANCE OF SCARF ROBUSTNESS IMPROVEMENT
# ================================================================

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score
)

print("=" * 80)
print("M7 — SCARF vs STANDARD: PAIRED BOOTSTRAP ANALYSIS")
print("=" * 80)

# ------------------------------------------------
# SETTINGS
# ------------------------------------------------

N_BOOT = 2000
SEED = 2026

rng = np.random.default_rng(SEED)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", DEVICE)
print("Bootstrap iterations:", N_BOOT)


# ------------------------------------------------
# CHECK MODELS
# ------------------------------------------------

if "standard_model_found" not in globals():
    raise RuntimeError(
        "standard_model_found not found. "
        "Run M6 first."
    )

if "scarf_model_found" not in globals():
    raise RuntimeError(
        "scarf_model_found not found. "
        "Run M6 first."
    )

if "test_df" not in globals():
    raise RuntimeError(
        "test_df not found."
    )

if "preprocessor" not in globals():
    raise RuntimeError(
        "preprocessor not found."
    )


# ------------------------------------------------
# TEST LABELS
# ------------------------------------------------

y = test_df["label"].astype(int).values

print("Test samples:", len(y))


# ------------------------------------------------
# PREDICTION FUNCTION
# ------------------------------------------------

def predict_probs(model, X):

    model.eval()

    X_tensor = torch.tensor(
        X,
        dtype=torch.float32
    ).to(DEVICE)

    outputs = []

    with torch.no_grad():

        for start in range(
            0,
            len(X_tensor),
            4096
        ):

            batch = X_tensor[
                start:start + 4096
            ]

            out = model(batch)

            if out.ndim > 1:
                out = out.squeeze(-1)

            prob = torch.sigmoid(out)

            outputs.append(
                prob.cpu().numpy()
            )

    return np.concatenate(outputs)


# ------------------------------------------------
# CREATE TEST DATA
# ------------------------------------------------

def make_dttl_test(percent):

    df = test_df.drop(
        columns=["label"]
    ).copy()

    if percent != 0:

        values = pd.to_numeric(
            df["dttl"],
            errors="coerce"
        ).astype(float)

        factor = 1 + percent / 100.0

        transformed = np.round(
            values * factor
        )

        transformed = np.clip(
            transformed,
            1,
            255
        )

        df["dttl"] = transformed

    X = preprocessor.transform(df)

    if hasattr(X, "toarray"):
        X = X.toarray()

    return np.asarray(
        X,
        dtype=np.float32
    )


# ------------------------------------------------
# METRIC FUNCTION
# ------------------------------------------------

def metrics_from_probs(y_true, probs):

    pred = (
        probs >= 0.5
    ).astype(int)

    tp = np.sum(
        (y_true == 1) &
        (pred == 1)
    )

    fn = np.sum(
        (y_true == 1) &
        (pred == 0)
    )

    tpr = (
        tp / (tp + fn)
        if (tp + fn) > 0
        else np.nan
    )

    auroc = roc_auc_score(
        y_true,
        probs
    )

    auprc = average_precision_score(
        y_true,
        probs
    )

    return tpr, auroc, auprc


# ================================================================
# 1. COLLECT PER-SAMPLE PREDICTIONS
# ================================================================

perturbations = [-15, -10, -5, -2, 0]

prediction_store = {}

print("\nCollecting predictions...")

for pct in perturbations:

    print(
        f"  dTTL {pct:+d}%"
    )

    X = make_dttl_test(pct)

    standard_probs = predict_probs(
        standard_model_found,
        X
    )

    scarf_probs = predict_probs(
        scarf_model_found,
        X
    )

    prediction_store[pct] = {
        "standard": standard_probs,
        "scarf": scarf_probs
    }


# ================================================================
# 2. PAIRED BOOTSTRAP
# ================================================================

def paired_bootstrap(
    y_true,
    standard_probs,
    scarf_probs,
    n_boot=2000
):

    n = len(y_true)

    observed_std = metrics_from_probs(
        y_true,
        standard_probs
    )

    observed_scarf = metrics_from_probs(
        y_true,
        scarf_probs
    )

    observed_difference = (
        np.array(observed_scarf)
        -
        np.array(observed_std)
    )

    bootstrap_differences = []

    for _ in range(n_boot):

        indices = rng.integers(
            0,
            n,
            size=n
        )

        y_b = y_true[indices]

        std_b = standard_probs[
            indices
        ]

        scarf_b = scarf_probs[
            indices
        ]

        try:

            std_metrics = metrics_from_probs(
                y_b,
                std_b
            )

            scarf_metrics = metrics_from_probs(
                y_b,
                scarf_b
            )

            diff = (
                np.array(scarf_metrics)
                -
                np.array(std_metrics)
            )

            bootstrap_differences.append(
                diff
            )

        except ValueError:
            continue

    bootstrap_differences = np.asarray(
        bootstrap_differences
    )

    ci_low = np.percentile(
        bootstrap_differences,
        2.5,
        axis=0
    )

    ci_high = np.percentile(
        bootstrap_differences,
        97.5,
        axis=0
    )

    return (
        observed_difference,
        ci_low,
        ci_high
    )


# ================================================================
# 3. RUN SIGNIFICANCE TEST
# ================================================================

significance_rows = []

for pct in perturbations:

    print(
        f"\nBootstrap testing dTTL {pct:+d}%..."
    )

    std_probs = prediction_store[
        pct
    ]["standard"]

    scarf_probs = prediction_store[
        pct
    ]["scarf"]

    difference, low, high = paired_bootstrap(
        y,
        std_probs,
        scarf_probs,
        N_BOOT
    )

    metrics_names = [
        "TPR",
        "AUROC",
        "AUPRC"
    ]

    for i, metric in enumerate(
        metrics_names
    ):

        significant = (
            low[i] > 0
            or
            high[i] < 0
        )

        significance_rows.append({

            "dTTL_%": pct,

            "Metric": metric,

            "SCARF_minus_Standard":
                difference[i],

            "CI_95_low":
                low[i],

            "CI_95_high":
                high[i],

            "Significant_95pct":
                significant
        })


significance_df = pd.DataFrame(
    significance_rows
)


# ================================================================
# 4. DISPLAY
# ================================================================

print("\n" + "=" * 80)
print("SCARF vs STANDARD — PAIRED BOOTSTRAP")
print("=" * 80)

print(
    significance_df.round(6).to_string(
        index=False
    )
)


# ================================================================
# 5. ROBUSTNESS-OF-DEGRADATION ANALYSIS
# ================================================================

print("\n" + "=" * 80)
print("ROBUSTNESS DEGRADATION ANALYSIS")
print("=" * 80)

clean_std = metrics_from_probs(
    y,
    prediction_store[0]["standard"]
)

clean_scarf = metrics_from_probs(
    y,
    prediction_store[0]["scarf"]
)

robustness_rows = []

for pct in [-2, -5, -10, -15]:

    std_metrics = metrics_from_probs(
        y,
        prediction_store[pct]["standard"]
    )

    scarf_metrics = metrics_from_probs(
        y,
        prediction_store[pct]["scarf"]
    )

    # TPR changes from each model's own clean state

    std_tpr_change = (
        std_metrics[0]
        -
        clean_std[0]
    )

    scarf_tpr_change = (
        scarf_metrics[0]
        -
        clean_scarf[0]
    )

    # Positive = SCARF loses less TPR
    robustness_gain = (
        abs(std_tpr_change)
        -
        abs(scarf_tpr_change)
    )

    relative_gain = (
        robustness_gain /
        abs(std_tpr_change)
        * 100
    )

    robustness_rows.append({

        "dTTL_%": pct,

        "Standard_TPR_change":
            std_tpr_change,

        "SCARF_TPR_change":
            scarf_tpr_change,

        "SCARF_less_degradation":
            robustness_gain,

        "Relative_reduction_in_degradation_%":
            relative_gain
    })


robustness_df = pd.DataFrame(
    robustness_rows
)

print(
    robustness_df.round(6).to_string(
        index=False
    )
)


# ================================================================
# 6. CLEAN PERFORMANCE COST
# ================================================================

print("\n" + "=" * 80)
print("CLEAN PERFORMANCE COST OF SCARF")
print("=" * 80)

print(
    f"Standard TPR  : {clean_std[0]:.6f}"
)

print(
    f"SCARF TPR     : {clean_scarf[0]:.6f}"
)

print(
    f"TPR difference: "
    f"{clean_scarf[0] - clean_std[0]:+.6f}"
)

print()

print(
    f"Standard AUROC  : {clean_std[1]:.6f}"
)

print(
    f"SCARF AUROC     : {clean_scarf[1]:.6f}"
)

print(
    f"AUROC difference: "
    f"{clean_scarf[1] - clean_std[1]:+.6f}"
)

print()

print(
    f"Standard AUPRC : {clean_std[2]:.6f}"
)

print(
    f"SCARF AUPRC    : {clean_scarf[2]:.6f}"
)

print(
    f"AUPRC difference: "
    f"{clean_scarf[2] - clean_std[2]:+.6f}"
)


# ================================================================
# 7. SAVE
# ================================================================

significance_df.to_csv(
    "M7_scarf_paired_bootstrap.csv",
    index=False
)

robustness_df.to_csv(
    "M7_scarf_robustness_degradation.csv",
    index=False
)

print("\nSaved:")
print("M7_scarf_paired_bootstrap.csv")
print("M7_scarf_robustness_degradation.csv")

print("\n" + "=" * 80)
print("M7 COMPLETE")
print("=" * 80)

M7 — SCARF vs STANDARD: PAIRED BOOTSTRAP ANALYSIS
Device: cuda
Bootstrap iterations: 2000
Test samples: 82332

  dTTL -15%
  dTTL -10%
  dTTL -5%
  dTTL -2%
  dTTL +0%

Bootstrap testing dTTL -15%...

Bootstrap testing dTTL -10%...

Bootstrap testing dTTL -5%...

Bootstrap testing dTTL -2%...

Bootstrap testing dTTL +0%...

SCARF vs STANDARD — PAIRED BOOTSTRAP
 dTTL_% Metric  SCARF_minus_Standard  CI_95_low  CI_95_high  Significant_95pct
    -15    TPR              0.031457   0.028972    0.034041               True
    -15  AUROC              0.143630   0.141386    0.145909               True
    -15  AUPRC              0.080025   0.078355    0.081717               True
    -10    TPR              0.031854   0.029319    0.034362               True
    -10  AUROC              0.140945   0.138665    0.143439               True
    -10  AUPRC              0.077778   0.076153    0.079485               True
     -5    TPR              0.032648   0.030264    0.035160               True
     